# 02 - Analise exploratoria de dados de RHAnalise exploratoria e a etapa em que conhecemos os dados antes de afirmar qualquer coisa. Vamos calcular indicadores, procurar padroes, criar graficos e praticar interpretacao.A pergunta constante deste notebook e: o que o dado mostra, o que ele nao mostra e qual proximo passo ele sugere?

In [ ]:
# Importamos pathlib para localizar arquivos.from pathlib import Path# pandas trabalha com tabelas.import pandas as pd# matplotlib e seaborn criam graficos.import matplotlib.pyplot as pltimport seaborn as sns# Definimos um estilo visual simples para os graficos.sns.set_theme(style="whitegrid")# Pedimos ao pandas para mostrar ate 60 colunas quando necessario.pd.set_option("display.max_columns", 60)# Localizamos a raiz do projeto e a pasta de dados.BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()DADOS = BASE_DIR / "dados"

## 1. Carregar bases principaisVamos abrir quatro bases. Cada uma tem uma unidade de analise diferente: vaga, treinamento, colaborador e mes.

In [ ]:
# Cada read_csv abre um arquivo e cria um DataFrame.vagas = pd.read_csv(DADOS / "vagas_recrutamento.csv")treinamentos = pd.read_csv(DADOS / "treinamentos.csv")colaboradores = pd.read_csv(DADOS / "colaboradores.csv")indicadores = pd.read_csv(DADOS / "indicadores_mensais.csv")# Mostramos o tamanho de cada tabela para entender o volume de dados.print(f"Vagas: {vagas.shape[0]} linhas e {vagas.shape[1]} colunas")print(f"Treinamentos: {treinamentos.shape[0]} linhas e {treinamentos.shape[1]} colunas")print(f"Colaboradores: {colaboradores.shape[0]} linhas e {colaboradores.shape[1]} colunas")print(f"Indicadores: {indicadores.shape[0]} linhas e {indicadores.shape[1]} colunas")

## 2. Checagem simples de qualidadeAntes de calcular KPIs, verificamos dados faltantes e duplicados. Em bases reais, esta etapa costuma consumir bastante tempo.

In [ ]:
# isna().sum() conta valores vazios por coluna.# duplicated().sum() conta linhas duplicadas completas.for nome, tabela in {    "vagas": vagas,    "treinamentos": treinamentos,    "colaboradores": colaboradores,    "indicadores": indicadores,}.items():    print(f"Base: {nome}")    print(f"Linhas duplicadas: {tabela.duplicated().sum()}")    print("Colunas com valores faltantes:")    print(tabela.isna().sum()[tabela.isna().sum() > 0])

## 3. Recrutamento: velocidade, custo e volumeUma fonte de contratacao nao deve ser avaliada por um unico indicador. Vamos olhar volume de vagas, contratados, time to hire e custo.

In [ ]:
# Agrupamos por source_of_hire para comparar fontes.kpis_recrutamento = (    vagas.groupby("source_of_hire")    .agg(        vagas=("vaga_id", "count"),        contratados=("contratados", "sum"),        time_to_hire_medio=("time_to_hire_dias", "mean"),        candidatos_por_vaga=("candidatos", "mean"),        custo_total=("custo_divulgacao", "sum"),    ))# Criamos um indicador derivado: custo por contratado.kpis_recrutamento["custo_por_contratado"] = (    kpis_recrutamento["custo_total"] / kpis_recrutamento["contratados"])# Arredondamos e ordenamos para facilitar leitura.kpis_recrutamento.round(1).sort_values("time_to_hire_medio")

In [ ]:
# Criamos uma figura com tamanho definido.plt.figure(figsize=(9, 4))# barplot mostra a media de time_to_hire_dias por source_of_hire.sns.barplot(data=vagas, x="source_of_hire", y="time_to_hire_dias", estimator="mean", errorbar=None)# Titulo e rotulos ajudam a pessoa leitora a entender o grafico sem adivinhar.plt.title("Time to hire medio por source of hire")plt.xlabel("Source of hire")plt.ylabel("Dias")plt.xticks(rotation=30, ha="right")plt.tight_layout()

## 4. Treinamento: conclusao, eficacia e custoTransformaremos `Sim` e `Nao` em 1 e 0 para calcular taxa de conclusao. Isso e comum em analises de indicadores.

In [ ]:
# map() troca valores de texto por numeros.# Sim vira 1; Nao vira 0.treinamentos["concluiu_bool"] = treinamentos["concluiu"].map({"Sim": 1, "Não": 0})# Agora agrupamos por modalidade.kpis_treinamento = (    treinamentos.groupby("modalidade")    .agg(        pessoas=("colaborador_id", "count"),        taxa_conclusao=("concluiu_bool", "mean"),        horas_medias=("horas_treinamento", "mean"),        nota_media=("nota_eficacia", "mean"),        custo_coffee_total=("custo_coffee", "sum"),    ))kpis_treinamento.round(2)

## 5. Headcount, representatividade e salariosAgora olhamos pessoas. Esta parte exige cuidado etico: genero e raca devem ser usados para medir representatividade e possiveis desigualdades, nao para rotular individuos.

In [ ]:
# value_counts conta quantas pessoas existem por area.headcount_area = colaboradores["area"].value_counts().rename_axis("area").reset_index(name="headcount")# normalize=True transforma contagem em proporcao.representatividade_genero = colaboradores["genero"].value_counts(normalize=True).mul(100).round(1)# Agrupamos salarios por nivel e calculamos estatisticas simples.salario_por_nivel = (    colaboradores.groupby("nivel")["salario_mensal"]    .agg(["count", "mean", "median", "min", "max"])    .round(0))headcount_area, representatividade_genero, salario_por_nivel

In [ ]:
# Boxplot mostra distribuicao, nao apenas media.# Ele ajuda a ver dispersao e valores mais extremos.plt.figure(figsize=(9, 4))sns.boxplot(data=colaboradores, x="nivel", y="salario_mensal")plt.title("Distribuicao salarial por nivel")plt.xlabel("Nivel")plt.ylabel("Salario mensal")plt.xticks(rotation=25, ha="right")plt.tight_layout()

## 6. Indicadores mensaisVamos calcular indicadores de turnover, absenteismo e retencao. Observe que cada formula tem numerador, denominador e periodo.

In [ ]:
# Headcount medio reduz distorcao quando ha admissoes e demissoes no mes.indicadores["headcount_medio"] = (indicadores["headcount_inicio"] + indicadores["headcount_fim"]) / 2# Turnover mensal: demitidos divididos pelo headcount medio.indicadores["turnover"] = indicadores["demitidos"] / indicadores["headcount_medio"]# Absenteismo: dias de ausencia divididos pelo total teorico de dias trabalhados.indicadores["taxa_absenteismo"] = indicadores["ausencias_dias"] / (    indicadores["headcount_fim"] * indicadores["dias_trabalho"])# Retencao por diversidade: 1 menos a taxa de desligamentos no grupo de diversidade.indicadores["retencao_diversidade"] = 1 - (    indicadores["desligamentos_diversidade"] / indicadores["headcount_diversidade"])indicadores[["mes", "turnover", "taxa_absenteismo", "retencao_diversidade", "corte_gastos_estimado"]].round(4)

In [ ]:
# Um grafico de linha ajuda a ver evolucao no tempo.plt.figure(figsize=(9, 4))sns.lineplot(data=indicadores, x="mes", y="turnover", marker="o", label="Turnover")sns.lineplot(data=indicadores, x="mes", y="taxa_absenteismo", marker="o", label="Absenteismo")plt.title("Turnover e absenteismo ao longo do tempo")plt.xlabel("Mes")plt.ylabel("Taxa")plt.xticks(rotation=45, ha="right")plt.tight_layout()

## 7. Exercicios de interpretacao1. Escolha um indicador de recrutamento e escreva uma conclusao.2. Escolha um indicador de treinamento e escreva uma limitacao.3. Explique por que salario medio sozinho nao prova equidade salarial.4. Identifique um mes que mereceria investigacao em turnover ou absenteismo.5. Escreva uma recomendacao pratica e nao punitiva para RH.